# Performance comparison

This  notebook compares the results of GWLR and GWRF

In [ ]:
import geopandas as gpd
import joblib
import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib_scalebar.scalebar import ScaleBar
from scipy import stats
from sklearn import metrics

In [ ]:
cluster_names = {
    1: "Incoherent Large-Scale Homogeneous Fabric",
    2: "Incoherent Large-Scale Heterogeneous Fabric",
    3: "Incoherent Small-Scale Linear Fabric",
    4: "Incoherent Small-Scale Sparse Fabric",
    5: "Incoherent Small-Scale Compact Fabric",
    6: "Coherent Interconnected Fabric",
    7: "Coherent Dense Disjoint Fabric",
    8: "Coherent Dense Adjacent Fabric",
}

results_lr = []

for cluster in [1, 3, 4, 5, 6, 7, 8]:
    model_path = f"/data/uscuni-restricted/06_models/fa/label_{cluster}/lr/model.joblib"
    with open(model_path, "rb") as f:
        model = joblib.load(f)

    # f1_values = model.local_pooled_f1_macro_
    f1_values = model.local_metric(metrics.f1_score, average="macro", zero_division=0)

    results_lr.append(
        {
            "cluster": cluster,
            "LR": np.nanmean(f1_values),
            #      "std_f1_macro_lr": np.nanstd(f1_values),
        }
    )

results_lr_df = pd.DataFrame(results_lr)

results_lr_df["cluster"] = results_lr_df["cluster"].replace(cluster_names)

results_lr_df = results_lr_df.set_index("cluster")

results_lr = results_lr_df.transpose()

In [ ]:
results_rf = []

for cluster in [1, 3, 4, 5, 6, 7, 8]:
    model_path = f"/data/uscuni-restricted/06_models/fa/label_{cluster}/rf/model.joblib"
    with open(model_path, "rb") as f:
        model = joblib.load(f)

    f1_values = model.local_metric(metrics.f1_score, average="macro")

    results_rf.append(
        {
            "cluster": cluster,
            "RF": np.nanmean(f1_values),
            #   "std_f1_macro_rf": np.nanstd(f1_values),
        }
    )

results_rf_df = pd.DataFrame(results_rf)

results_rf_df["cluster"] = results_rf_df["cluster"].replace(cluster_names)

results_rf_df = results_rf_df.set_index("cluster")


results_rf = results_rf_df.transpose()

In [ ]:
results = pd.concat([results_lr, results_rf], axis=0)


results.round(2).style.format("{:.2f}").background_gradient(
    cmap="GnBu", vmin=0.5, vmax=0.88
)

# Explore the importance of coefficients

In [ ]:
fi = {}
lc = {}
perf = []
perf_lr = []
perf_rf = []

for reduction in ["fa"]:
    fi[reduction] = {}
    lc[reduction] = {}
    for model_type in ["lr", "rf"]:
        fi[reduction][model_type] = {}
        lc[reduction][model_type] = {}
        for cluster in [1, 3, 4, 5, 6, 7, 8]:
            with open(
                f"/data/uscuni-restricted/06_models/{reduction}/label_{cluster}/{model_type}/model.joblib",
                "rb",
            ) as f:
                model = joblib.load(f)
                if model_type == "rf":
                    fi[reduction][model_type][cluster] = model.feature_importances_
                    perf_rf.append(
                        pd.Series(
                            {
                                "cluster": cluster,
                                "pooled_f1_macro": metrics.f1_score(
                                    model.oob_y_pooled_,
                                    model.oob_pred_pooled_,
                                    average="macro",
                                ),
                            }
                        )
                    )

                else:
                    lc[reduction][model_type][cluster] = model.local_coef_
                    perf_lr.append(
                        pd.Series(
                            {
                                "cluster": cluster,
                                "pooled_f1_macro": metrics.f1_score(
                                    model.y_pooled_, model.pred_pooled_, average="macro"
                                ),
                            }
                        )
                    )

                perf.append(
                    pd.Series(
                        {
                            "reduction": reduction,
                            "model": model_type,
                            "cluster": cluster,
                            # "accuracy": model.score_,
                            # "balanced_accuracy": model.balanced_accuracy_,
                            # "precision": model.precision_,
                            # "recall": model.recall_,
                            # "f1_macro": model.f1_macro_,
                            # "f1_micro_": model.f1_micro_,
                            # "f1_weighted": model.f1_weighted_,
                        }
                    )
                )
performance = pd.DataFrame(perf)
performance_lr = pd.DataFrame(perf_lr).set_index("cluster")
performance_rf = pd.DataFrame(perf_rf).set_index("cluster")

# Local coefficients


In [ ]:
rename_index = {
    "Obyvatelstvo - věk: 0 - 6  - celkem": "Age Group: 0-6",
    "Obyvatelstvo - věk: 7 - 14  - celkem": "Age Group: 7-14",
    "Obyvatelstvo - věk: 15 - 24  - celkem": "Age Group: 15-24",
    "Obyvatelstvo - věk: 45 - 54  - celkem": "Age Group: 45-54",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední vč. vyučení bez maturity - celkem": "Education - Secondary (without graduation)",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední s maturitou vč. nástavbového a pomaturitního - celkem": "Education - Secondary (with graduation)",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání:  vysokoškolské - celkem": "Education - University",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: nezjištěno - celkem": "Unknown Education",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem": "Empl. sector - Agriculture/Forestry/Fishery",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: průmysl - celkem": "Empl. sector - Industry",
    "Zaměstnaní - Pracovníci ve službách a prodeji": "Empl. occupation - Service/Sales",
    "Zaměstnaní - Řemeslníci a opraváři": "Empl. occupation - Craft/Repair",
    "Obyvatelstvo - zaměstnaní - postavení v zaměstnání: zaměstnanci - celkem": "Empl. status - Employees",
    "Obyvatelstvo - ekon. aktivita: osoby na rodičovské dovolené - celkem": "Econ. Activity - Parental Leave",
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví": "Residents in Owner-Occupied Dwellings",
    "Počet osob v bytech celkem  s právním důvodem užívání: nájemní / pronajatý": "Residents in Rented Dwelling",
    "Počet osob v bytech celkem  s právním důvodem užívání: družstevní": "Residents in Cooperative Dwellings",
    "Počet obyvatel na byt": "Persons per Dwelling",
    "Počet osob v domech celkem s vlastnictvím:  fyzická osoba": "Residents in Privately Owned Houses",
    "Obyvatelstvo - státní občanství: Slovenská republika - celkem": "Citizenship - Slovakia",
    "Obyvatelstvo - státní občanství: země EU mimo ČR - celkem": "Citizenship - EU",
    "Obyvatelstvo - státní občanství: nezjištěno - celkem": "Citizenship - Unknown",
    "Obyvatelstvo - náboženská víra: bez náboženské víry - celkem": "Religion - Non-Religious",
    "Obyvatelstvo - náboženská víra: neuvedeno - celkem": "Religion - Unspecified",
    "Obyvatelstvo - s trvalým pobytem - celkem": "Long-Term Stay Residents",
    "Obyvatelstvo - rodinný stav: ženatí, vdané - celkem": "Marital Status - Married",
    "Obyvatelstvo - rodinný stav: rozvedení - celkem": "Marital Status - Divorced",
    "Obyvatelstvo - rodinný stav: ovdovělí - celkem": "Marital Status - Widowed",
}
rename_columns = {
    1: "Incoherent Large-Scale Homogeneous Fabric",
    2: "Incoherent Large-Scale Heterogeneous Fabric",
    3: "Incoherent Small-Scale Linear Fabric",
    4: "Incoherent Small-Scale Sparse Fabric",
    5: "Incoherent Small-Scale Compact Fabric",
    6: "Coherent Interconnected Fabric",
    7: "Coherent Dense Disjoint Fabric",
    8: "Coherent Dense Adjacent Fabric",
}

In [ ]:
lc_abs_means = {}
lc_means = {}
lc_stds = {}

for k, v in lc["fa"]["lr"].items():
    if k in [1, 3, 4, 5, 6, 7, 8]:
        lc_abs_means[k] = v.abs().mean()
        lc_means[k] = v.mean()
        lc_stds[k] = v.std()

lc_abs_means = pd.DataFrame(lc_abs_means).rename(
    columns=rename_columns, index=rename_index
)
lc_means = pd.DataFrame(lc_means).rename(columns=rename_columns, index=rename_index)
lc_stds = pd.DataFrame(lc_stds).rename(columns=rename_columns, index=rename_index)

Get the variables with highest importance across all built form type

In [ ]:
# sort by largest absolute mean across models and
mask = (lc_abs_means > lc_abs_means.stack().quantile(0.50)).any(axis=1)
lc_abs_means_filtered = lc_abs_means.loc[mask]
row_abs_mean1 = lc_abs_means_filtered.mean(axis=1)
lc_abs_means_sorted = lc_abs_means_filtered.loc[
    row_abs_mean1.sort_values(ascending=False).index
]

lc_abs_means_sorted.head(7).style.format("{:.4f}").background_gradient(
    cmap="YlGnBu", vmin=0.02, vmax=0.8
)

In [ ]:
# sort by largest absolute mean across models and
mask = (lc_means > lc_means.stack().quantile(0.50)).any(axis=1)
lc_means_filtered = lc_means.loc[mask]
row_abs_mean2 = lc_means.mean(axis=1)
lc_means_sorted = lc_means.loc[
    row_abs_mean2.sort_values(ascending=False).index
].transpose()
lc_means_sorted = lc_means_sorted.transpose()

lc_means_sorted.head(10).style.format("{:.2f}").background_gradient(
    cmap="coolwarm", vmin=-0.7, vmax=0.7
)

Get their standard deviation

In [ ]:
lc_stds_filtered = lc_stds.loc[lc_abs_means_sorted.index]
# row_means = lc_stds_filtered.mean(axis=1)
lc_stds_filtered.head(10).style.format("{:.2f}").background_gradient(
    cmap="YlOrRd", vmin=0.2, vmax=0.4
)

# Plot the differences

In [ ]:
cmap = [
    "#4069BC",
    "#E69C63",
    "#eec1d5",
    "#E0665F",
    "#ECBF43",
    "#b2cd32",
    "#1F943E",
]
label = [
    "Incoherent Large-Scale\nHomogeneous Fabric",
    "Incoherent Small-Scale\nLinear Fabric",
    "Incoherent Small-Scale\nSparse Fabric",
    "Incoherent Small-Scale\nCompact Fabric",
    "Coherent Inter-\nconnected Fabric",
    "Coherent Dense\nDisjoint Fabric",
    "Coherent Dense\nAdjacent Fabric",
]

In [ ]:
jitter = 0.04
x_data = [
    np.array([i] * len(lc_stds_filtered))
    for i, d in enumerate(lc_stds_filtered.columns)
]
x_jittered = [x + stats.t(df=6, scale=jitter).rvs(len(x)) for x in x_data]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

medianprops = dict(linewidth=2, color="k", solid_capstyle="butt")
boxprops = dict(linewidth=1, color="k")

ax.boxplot(
    lc_stds_filtered,
    positions=range(len(lc_stds_filtered.columns)),
    showfliers=False,
    showcaps=False,
    tick_labels=label,
    medianprops=medianprops,
    whiskerprops=boxprops,
    boxprops=boxprops,
)

for x, col in enumerate(lc_stds_filtered.columns):
    ax.scatter(x_jittered[x], lc_stds_filtered[col], c=cmap[x], s=50, alpha=0.7)


ax.tick_params(axis="x", pad=6)

for lab in ax.get_xticklabels():
    lab.set_rotation(45)
    lab.set_ha("right")
    lab.set_rotation_mode("anchor")
sns.despine()
plt.savefig("boxplot-form.png", dpi=300, bbox_inches="tight")

Get tha variables that have both high feature importance and high standard deviation

In [ ]:
pd.DataFrame((lc_stds.std(axis=0).sort_values(ascending=False)).round(4))

Find out which variables have the highest stds across built fabrics

In [ ]:
dfs_mean = []
dfs_std = []
dfs_a_mean = []
dfs_a_std = []

for i in list(lc["fa"]["lr"].keys()):
    coefs = lc["fa"]["lr"][i]

    # Raw mean and std
    mean = coefs.mean(axis=0)
    std = coefs.std(axis=0)

    # Absolute mean and std
    abs_mean = abs(coefs).mean(axis=0)
    abs_std = abs(coefs).std(axis=0)

    # Create DataFrames
    df_mean = pd.DataFrame(mean, columns=[str(i)])
    df_std = pd.DataFrame(std, columns=[str(i)])
    df_a_mean = pd.DataFrame(abs_mean, columns=[str(i)])
    df_a_std = pd.DataFrame(abs_std, columns=[str(i)])

    # Append to lists
    dfs_mean.append(df_mean)
    dfs_std.append(df_std)
    dfs_a_mean.append(df_a_mean)
    dfs_a_std.append(df_a_std)

# Concatenate DataFrames
raw_mean = pd.concat(dfs_mean, axis=1)
raw_std = pd.concat(dfs_std, axis=1)
abs_mean = pd.concat(dfs_a_mean, axis=1)
abs_std = pd.concat(dfs_a_std, axis=1)

# Calculate overall mean and std across models
result = pd.DataFrame(
    {
        "Raw_Mean": raw_mean.mean(axis=1).round(2),
        "Raw_Std": raw_std.mean(axis=1).round(2),
        "Abs_Mean": abs_mean.mean(axis=1).round(2),
        "Abs_Std": abs_std.mean(axis=1).round(2),
    }
)

In [ ]:
result[["Abs_Mean", "Raw_Std"]].sort_values(by="Raw_Std", ascending=False).head(7)

For each fabric, get the most spatially homogenous and heterogenous variables

In [ ]:
# Filter for impactful variables
min_abs_coef = lc_means.stack().abs().quantile(0.5)

# Collect all standard deviations across the entire dataset
all_stds = []
for col in lc_means.columns:
    for var in lc_means.index:
        if abs(lc_means.loc[var, col]) > min_abs_coef:
            all_stds.append(lc_stds.loc[var, col])

# Define thresholds directly from the distribution of all STDs
quantiles_under = np.quantile(all_stds, 0.1)
quantiles_over = np.quantile(all_stds, 0.9)

# Process each fabric type
top_n = 10
extremes_by_bf = {}

for col in lc_means.columns:
    records = []
    for var in lc_means.index:
        coef = lc_means.loc[var, col]
        std = lc_stds.loc[var, col]
        abs_coef = lc_abs_means.loc[var, col]

        if abs(coef) > min_abs_coef:
            records.append((var, coef, std, abs_coef))

    # Sort and filter by the thresholds
    most_homo = [r for r in records if r[2] < quantiles_under]
    most_hetero = [r for r in records if r[2] > quantiles_over]

    # Rank results by impact (|coef|)
    extremes_by_bf[col] = {
        "homo": sorted(most_homo, key=lambda x: abs(x[1]), reverse=True)[:top_n],
        "hetero": sorted(most_hetero, key=lambda x: abs(x[1]), reverse=True)[:top_n],
    }

In [ ]:
# Print results with counts and averages
for bf_type, groups in extremes_by_bf.items():
    print(f"\n{bf_type}")

    for group_type in ["homo", "hetero"]:
        vars_list = groups[group_type]
        count = len(vars_list)

        print(f"  Most {group_type}geneous (count={count}):")

        if vars_list:
            # Calculate aggregate stats for this specific group
            avg_coef = sum(r[1] for r in vars_list) / count
            avg_std = sum(r[2] for r in vars_list) / count
            avg_abs_coef = sum(r[3] for r in vars_list) / count

            for var, coef, std, abs_coef in vars_list:
                print(
                    f"    {var}: mean coef={coef:.3f}, std={std:.3f}, abs_coef={abs_coef:.3f}"
                )

            print(
                f"    → Averages: mean={avg_coef:.3f}, std={avg_std:.3f}, abs_coef={avg_abs_coef:.3f}"
            )
        else:
            print("    None above/below threshold")

In [ ]:
lc_mean = pd.concat(lc["fa"]["lr"])

In [ ]:
lc_mean = lc_mean.rename(columns=rename_index)

In [ ]:
var_category = {
    # Demographics
    "Age Group: 0-6": "Demographics",
    "Age Group: 7-14": "Demographics",
    "Age Group: 15-24": "Demographics",
    "Age Group: 45-54": "Demographics",
    "Citizenship - Slovakia": "Demographics",
    "Citizenship - EU": "Demographics",
    "Citizenship - Unknown": "Demographics",
    "Long-Term Stay Residents": "Demographics",
    "Religion - Non-Religious": "Demographics",
    "Religion - Unspecified": "Demographics",
    "Marital Status - Married": "Demographics",
    "Marital Status - Divorced": "Demographics",
    "Marital Status - Widowed": "Demographics",
    # Socioeconomic
    "Education - Secondary (without graduation)": "Socioeconomic",
    "Education - Secondary (with graduation)": "Socioeconomic",
    "Education - University": "Socioeconomic",
    "Unknown Education": "Socioeconomic",
    "Empl. sector - Agriculture/Forestry/Fishery": "Socioeconomic",
    "Empl. sector - Industry": "Socioeconomic",
    "Empl. occupation - Service/Sales": "Socioeconomic",
    "Empl. occupation - Craft/Repair": "Socioeconomic",
    "Empl. status - Employees": "Socioeconomic",
    "Econ. Activity - Parental Leave": "Socioeconomic",
    # Housing
    "Residents in Owner-Occupied Dwellings": "Housing",
    "Residents in Rented Dwelling": "Housing",
    "Residents in Cooperative Dwellings": "Housing",
    "Residents in Privately Owned Houses": "Housing",
    "Persons per Dwelling": "Housing",
}


category_palette = {
    "Demographics": "#b2cd32",
    "Socioeconomic": "#7CBAE4",
    "Housing": "#ECBF43",
}

In [ ]:
lc_mean_m = lc_mean.abs().mean(axis=0)

lc_mean_m.sort_values(ascending=False).head(2)

In [ ]:
ordered_cols = lc_mean_m.sort_values(ascending=False).index

palette = [category_palette[var_category[col]] for col in ordered_cols]
jitter = 0.06

# Get the sorted column order
sorted_cols = lc_mean_m.sort_values(ascending=False).index

# Create x_data and x_jittered for the sorted columns
x_data = [np.array([i] * len(lc_mean)) for i, _ in enumerate(sorted_cols)]
x_jittered = [x + stats.t(df=6, scale=jitter).rvs(len(x)) for x in x_data]

fig, ax = plt.subplots(figsize=(8, 4))

medianprops = dict(linewidth=1.2, color="k", solid_capstyle="butt")
boxprops = dict(linewidth=0.8, color="k")

ax.boxplot(
    lc_mean.loc[:, sorted_cols].dropna(),
    positions=range(len(sorted_cols)),
    showfliers=False,
    showcaps=False,
    tick_labels=sorted_cols,
    medianprops=medianprops,
    whiskerprops=boxprops,
    boxprops=boxprops,
)

for x, col in enumerate(sorted_cols):
    ax.scatter(x_jittered[x], lc_mean[col], c=palette[x], s=0.005, alpha=0.1)

ax.tick_params(axis="x")
for lab in ax.get_xticklabels():
    lab.set_rotation(90)
    lab.set_ha("right")
    lab.set_rotation_mode("anchor")

legend_handles = [
    mpatches.Patch(color=color, label=label)
    for label, color in category_palette.items()
]

# 3. Add the legend to the axes
ax.legend(
    handles=legend_handles,
    bbox_to_anchor=(0.5, -1.1),  # Moves legend outside the plot
    loc="center",
    ncol=3,
    frameon=False,
)

sns.despine()
plt.savefig("boxplot.png", dpi=300, bbox_inches="tight")

In [ ]:
lc_mean_std = lc_mean.std(axis=0)
lc_mean_std.sort_values(ascending=False).head(2)

In [ ]:
dfs = []

for i in list(lc["fa"]["lr"].keys()):
    imp = lc["fa"]["lr"][i].abs().mean(axis=0)
    imp = pd.DataFrame(imp, columns=[str(i)])
    dfs.append(imp)

abs_mean = pd.concat(dfs, axis=1)
## nejvyšší vliv mají obecně tyhle proměnný

imp = pd.DataFrame(abs_mean.mean(axis=1).sort_values(ascending=False).round(4))

In [ ]:
imp.loc[imp[0] > imp[0].quantile(0.75)]

In [ ]:
# jak se liší vliv těch proměnnejch
dfs = []

for i in list(lc["fa"]["lr"].keys()):
    imp = lc["fa"]["lr"][i].abs().std(axis=0)
    imp = pd.DataFrame(imp, columns=[str(i)])
    dfs.append(imp)

abs_mean = pd.concat(dfs, axis=1)
## nejvyšší vliv mají obecně tyhle proměnný

imp = pd.DataFrame(abs_mean.mean(axis=1).sort_values(ascending=False).round(4))
imp.loc[imp[0] > imp[0].quantile(0.75)]

In [ ]:
# u jakejch proměnnejch se nejvíc liší směr
dfs = []

for i in list(lc["fa"]["lr"].keys()):
    imp = lc["fa"]["lr"][i].std(axis=0)
    imp = pd.DataFrame(imp, columns=[str(i)])
    dfs.append(imp)

abs_mean = pd.concat(dfs, axis=1)
## nejvyšší vliv mají obecně tyhle proměnný

imp = pd.DataFrame(abs_mean.mean(axis=1).sort_values(ascending=False).round(4))
imp.loc[imp[0] > imp[0].quantile(0.75)]

In [ ]:
lc_mean = pd.concat(lc["fa"]["lr"]).groupby(level=1).mean()

In [ ]:
lc_std = pd.concat(lc["fa"]["lr"]).groupby(level=1).std()

# Maps

In [ ]:
census = gpd.read_parquet(
    "/data/uscuni-restricted/04_spatial_census/_merged_census_2021_relative_scaled.parquet"
)

selection = [
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední vč. vyučení bez maturity - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední s maturitou vč. nástavbového a pomaturitního - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání:  vysokoškolské - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: nezjištěno - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: průmysl - celkem",
    "Zaměstnaní - Pracovníci ve službách a prodeji",
    "Zaměstnaní - Řemeslníci a opraváři",
    "Obyvatelstvo - zaměstnaní - postavení v zaměstnání: zaměstnanci - celkem",
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    "Počet osob v bytech celkem  s právním důvodem užívání: nájemní / pronajatý",
    "Počet osob v bytech celkem  s právním důvodem užívání: družstevní",
    "Počet obyvatel na byt",
    "Počet osob v domech celkem s vlastnictvím:  fyzická osoba",
    "Obyvatelstvo - věk: 0 - 6  - celkem",
    "Obyvatelstvo - věk: 7 - 14  - celkem",
    "Obyvatelstvo - věk: 15 - 24  - celkem",
    "Obyvatelstvo - věk: 45 - 54  - celkem",
    "Obyvatelstvo - ekon. aktivita: osoby na rodičovské dovolené - celkem",
    "Obyvatelstvo - státní občanství: Slovenská republika - celkem",
    "Obyvatelstvo - státní občanství: země EU mimo ČR - celkem",
    "Obyvatelstvo - státní občanství: nezjištěno - celkem",
    "Obyvatelstvo - náboženská víra: bez náboženské víry - celkem",
    "Obyvatelstvo - náboženská víra: neuvedeno - celkem",
    "Obyvatelstvo - s trvalým pobytem - celkem",
    "Obyvatelstvo - rodinný stav: ženatí, vdané - celkem",
    "Obyvatelstvo - rodinný stav: rozvedení - celkem",
    "Obyvatelstvo - rodinný stav: ovdovělí - celkem",
    "geometry",
]
fas = census[selection]
clusters = pd.read_csv(
    "/data/uscuni-restricted/04_spatial_census/cluster_assignment_v10.csv",
    dtype={"kod_nadzsj_d": str},
)
cluster_mapping = pd.read_parquet(
    "/data/uscuni-ulce/processed_data/clusters/cluster_mapping_v10.pq"
)
data = fas.merge(clusters, left_on="nadzsjd", right_on="kod_nadzsj_d")
variables = data.columns.drop(["geometry", "kod_nadzsj_d", "final_without_noise"])

mapped = data["final_without_noise"].map(cluster_mapping[3])

In [ ]:
models = []

for reduction in ["fa"]:
    for model_type in ["lr"]:
        for cluster in [5, 4]:
            path = f"/data/uscuni-restricted/06_models/{reduction}/label_{cluster}/{model_type}/model.joblib"
            with open(path, "rb") as f:
                model = joblib.load(f)
            f1_values = model.local_metric(
                metrics.f1_score, average="macro", zero_division=0
            )

            models.append(
                {
                    "reduction": reduction,
                    "model_type": model_type,
                    "cluster": cluster,
                    "model": model,
                    "f1_values": model.local_metric(
                        metrics.f1_score, average="macro", zero_division=0
                    ),
                }
            )

df_models = pd.DataFrame(models)

In [ ]:
coef_cols = [
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
]

n_clusters = len(df_models)

# global scaling per variable
global_max_dict = {
    col: max(row["model"].local_coef_[col].max() for _, row in df_models.iterrows())
    for col in coef_cols
}

global_min_dict = {
    col: max(row["model"].local_coef_[col].min() for _, row in df_models.iterrows())
    for col in coef_cols
}

In [ ]:
fig, axes = plt.subplots(n_clusters, len(coef_cols), figsize=(12, 8))


# loop over clusters and variables
for row_i, (_, row) in enumerate(df_models.iterrows()):
    model = row["model"]
    cluster = row["cluster"]

    for col_i, coef_col in enumerate(coef_cols):
        ax = axes[col_i, row_i]

        series = model.local_coef_[coef_col]
        tmp = data.assign(_coef_tmp=series.values)

        tmp.plot(
            column="_coef_tmp",
            ax=ax,
            cmap="coolwarm",
            vmin=-1,
            vmax=1,
            legend=False,
            missing_kwds={"color": "lightgray"},
        )
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

axes[0][0].set_title("Incoherent Small-Scale Compact Fabric", fontsize=12)
axes[0][0].set_ylabel("Res. in Owner-Occupied Dwellings", fontsize=10)
axes[0][1].set_title("Incoherent Small-Scale Sparse Fabric", fontsize=12)
axes[1][0].set_ylabel("Empl. sector - Agriculture/Forestry/Fishery", fontsize=10)
axes[1][0].add_artist(ScaleBar(1, location="lower left", height_fraction=0.015))


# Colorbar in col 4, row 2 (last subplot)
cax = fig.add_axes((0.95, 0.2, 0.02, 0.6))
sm = mpl.cm.ScalarMappable(cmap="coolwarm", norm=mpl.colors.Normalize())
sm.set_array([-1, 1])
fig.colorbar(sm, cax=cax)
# plt.tight_layout()
fig.subplots_adjust(wspace=0, hspace=0)
# fig.align_titles()

# plt.tight_layout()
fig.savefig("side_by_side.png", dpi=300, bbox_inches="tight")